# Notebook: Apache Spark com Apache Iceberg

Notebook completo com modelagem ER, DDL, operações DML e snapshots.

## 1) Apache Iceberg

Iceberg é um formato aberto de tabela para data lake com ACID e snapshots.

In [48]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

if 'spark' in globals():
    spark.stop()


project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = project_root / 'data' / 'vendas.csv'
warehouse_path = project_root / 'data' / 'iceberg' / 'warehouse'

if warehouse_path.exists():
    import shutil
    shutil.rmtree(warehouse_path)

warehouse_uri = 'file://' + str(warehouse_path.resolve())
    
spark = SparkSession.builder.appName('ProjetoIceberg').master('local[*]') \
        .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1') \
        .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
        .config('spark.sql.catalog.demo', 'org.apache.iceberg.spark.SparkCatalog') \
        .config('spark.sql.catalog.demo.type', 'hadoop') \
        .config('spark.sql.catalog.demo.warehouse', warehouse_uri) \
        .getOrCreate()


print('Spark:', spark.version)
print('Warehouse:', warehouse_uri)

Spark: 3.5.1
Warehouse: file:///home/dev-luiz/Documentos/GitHub/Apache-Spark/data/iceberg/warehouse


## 2) Leitura do CSV

In [34]:
df_raw = spark.read.option('header', True).option('inferSchema', True).csv(str(data_path))
df = (df_raw
      .withColumn('data_pedido', to_date(col('data_pedido'), 'yyyy-MM-dd'))
      .withColumn('quantidade', col('quantidade').cast('int'))
      .withColumn('valor_unitario', col('valor_unitario').cast('double'))
      .withColumn('valor_total', col('valor_total').cast('double')))
df.createOrReplaceTempView('vendas_staging')
df.show(5, truncate=False)

+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|id_pedido|data_pedido|cliente         |produto           |categoria     |quantidade|valor_unitario|valor_total|status  |
+---------+-----------+----------------+------------------+--------------+----------+--------------+-----------+--------+
|1        |2024-11-23 |Daniel Rocha    |Fone Bluetooth Pro|Eletrônicos   |2         |287.96        |575.92     |pendente|
|2        |2024-12-12 |Yasmin Oliveira |Secador Íon       |Beleza        |5         |248.89        |1244.45    |pago    |
|3        |2024-04-21 |Henrique Martins|Escova Rotativa   |Beleza        |1         |183.68        |183.68     |pendente|
|4        |2024-12-25 |Rafael Cardoso  |Cafeteira Elétrica|Casa e Cozinha|4         |296.53        |1186.12    |pendente|
|5        |2024-01-04 |Zeca Antunes    |Fundamentos de SQL|Livros        |3         |72.27         |216.81     |pago    |
+---------+-----------+-

## 3) Modelagem ER

```text
+----------------------------+
|       PEDIDOS_ICEBERG      |
+----------------------------+
| id_pedido (PK lógico)      |
| data_pedido                |
| cliente                    |
| produto                    |
| categoria                  |
| quantidade                 |
| valor_unitario             |
| valor_total                |
| status                     |
+----------------------------+
```

## 4) DDL

```sql
CREATE TABLE demo.ecommerce.pedidos_iceberg (...) USING iceberg
PARTITIONED BY (months(data_pedido), categoria)
TBLPROPERTIES ('format-version' = '2');
```

In [35]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS demo.ecommerce')
spark.sql('DROP TABLE IF EXISTS demo.ecommerce.pedidos_iceberg')
spark.sql("CREATE TABLE demo.ecommerce.pedidos_iceberg (id_pedido INT, data_pedido DATE, cliente STRING, produto STRING, categoria STRING, quantidade INT, valor_unitario DOUBLE, valor_total DOUBLE, status STRING) USING iceberg PARTITIONED BY (months(data_pedido), categoria) TBLPROPERTIES ('format-version'='2')")
spark.sql('INSERT INTO demo.ecommerce.pedidos_iceberg SELECT id_pedido, data_pedido, cliente, produto, categoria, quantidade, valor_unitario, valor_total, status FROM vendas_staging')
spark.sql('SELECT COUNT(*) AS total_registros FROM demo.ecommerce.pedidos_iceberg').show()

+---------------+
|total_registros|
+---------------+
|            100|
+---------------+



## 5) INSERT

In [36]:
spark.sql("INSERT INTO demo.ecommerce.pedidos_iceberg VALUES (2001, DATE'2024-12-15', 'Patrícia Almeida', 'Tablet 10', 'Eletrônicos', 1, 1299.90, 1299.90, 'pendente')")
spark.sql("INSERT INTO demo.ecommerce.pedidos_iceberg VALUES (2002, DATE'2024-12-16', 'Roberto Neves', 'Livro Arquitetura de Dados', 'Livros', 1, 129.90, 129.90, 'pago')")
spark.sql('SELECT * FROM demo.ecommerce.pedidos_iceberg WHERE id_pedido >= 2001').show(truncate=False)

+---------+-----------+----------------+--------------------------+-----------+----------+--------------+-----------+--------+
|id_pedido|data_pedido|cliente         |produto                   |categoria  |quantidade|valor_unitario|valor_total|status  |
+---------+-----------+----------------+--------------------------+-----------+----------+--------------+-----------+--------+
|2002     |2024-12-16 |Roberto Neves   |Livro Arquitetura de Dados|Livros     |1         |129.9         |129.9      |pago    |
|2001     |2024-12-15 |Patrícia Almeida|Tablet 10                 |Eletrônicos|1         |1299.9        |1299.9     |pendente|
+---------+-----------+----------------+--------------------------+-----------+----------+--------------+-----------+--------+



## 6) UPDATE

In [37]:
spark.sql("UPDATE demo.ecommerce.pedidos_iceberg SET status = 'pago' WHERE id_pedido = 2001")
spark.sql('SELECT id_pedido, status FROM demo.ecommerce.pedidos_iceberg WHERE id_pedido = 2001').show()

+---------+------+
|id_pedido|status|
+---------+------+
|     2001|  pago|
+---------+------+



## 7) DELETE

In [38]:
antes = spark.sql("SELECT COUNT(*) AS qtd FROM demo.ecommerce.pedidos_iceberg WHERE status = 'cancelado' AND id_pedido <= 100").collect()[0]['qtd']
spark.sql("DELETE FROM demo.ecommerce.pedidos_iceberg WHERE status = 'cancelado' AND id_pedido <= 100")
depois = spark.sql("SELECT COUNT(*) AS qtd FROM demo.ecommerce.pedidos_iceberg WHERE status = 'cancelado' AND id_pedido <= 100").collect()[0]['qtd']
print('Cancelados antes:', antes, '| depois:', depois)

Cancelados antes: 17 | depois: 0


## 8) SELECT com filtros

In [39]:
spark.sql("SELECT categoria, status, COUNT(*) AS pedidos, ROUND(SUM(valor_total), 2) AS faturamento FROM demo.ecommerce.pedidos_iceberg WHERE data_pedido BETWEEN DATE'2024-01-01' AND DATE'2024-12-31' GROUP BY categoria, status ORDER BY faturamento DESC").show(50, truncate=False)

+--------------+--------+-------+-----------+
|categoria     |status  |pedidos|faturamento|
+--------------+--------+-------+-----------+
|Eletrônicos   |pago    |14     |23036.8    |
|Eletrônicos   |pendente|5      |12808.65   |
|Casa e Cozinha|pago    |10     |11451.16   |
|Beleza        |pago    |14     |7770.09    |
|Casa e Cozinha|pendente|7      |5952.83    |
|Livros        |pago    |16     |4736.88    |
|Roupas        |pago    |10     |3806.96    |
|Roupas        |pendente|4      |2043.06    |
|Beleza        |pendente|3      |1115.14    |
|Livros        |pendente|2      |490.41     |
+--------------+--------+-------+-----------+



## 9) Time Travel / Snapshots

In [40]:
snaps = spark.sql('SELECT snapshot_id, committed_at, operation FROM demo.ecommerce.pedidos_iceberg.snapshots ORDER BY committed_at')
snaps.show(truncate=False)
first_snapshot = snaps.collect()[0]['snapshot_id']
print('Snapshot inicial:', first_snapshot)
spark.sql('SELECT id_pedido, status FROM demo.ecommerce.pedidos_iceberg VERSION AS OF ' + str(first_snapshot) + ' ORDER BY id_pedido LIMIT 10').show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|6621327982898485908|2026-04-28 20:08:27.063|append   |
|1132836065302210890|2026-04-28 20:08:27.431|append   |
|1183059540872202025|2026-04-28 20:08:27.529|append   |
|1189905765834690468|2026-04-28 20:08:29.068|overwrite|
|1856911751646436300|2026-04-28 20:08:30.176|overwrite|
+-------------------+-----------------------+---------+

Snapshot inicial: 6621327982898485908
+---------+---------+
|id_pedido|status   |
+---------+---------+
|1        |pendente |
|2        |pago     |
|3        |pendente |
|4        |pendente |
|5        |pago     |
|6        |pago     |
|7        |cancelado|
|8        |pendente |
|9        |pendente |
|10       |pago     |
+---------+---------+



## 10) ACID

- Atomicidade
- Consistência
- Isolamento
- Durabilidade

In [41]:
spark.stop()
print('Sessão Spark encerrada.')

Sessão Spark encerrada.
